# Predviđanje cijena kuća — Modeli i Evaluacija

**Završni projekt — Strojno i duboko učenje**

U ovom notebooku treniramo tri modela različite složenosti, evaluiramo ih na validacijskom skupu i objašnjavamo predviđanja koristeći SHAP.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
import shap

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)

## 1. Učitavanje predprocesiranih podataka

Učitavamo podatke pripremljene u notebooku `01_EDA_preprocessing.ipynb`.

In [ ]:
with open('../output/results/preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train      = data['X_train']
X_val        = data['X_val']
y_train      = data['y_train']
y_val        = data['y_val']
X_test_final = data['X_test_final']

print('X_train:', X_train.shape, '| X_val:', X_val.shape)

## 2. Funkcija za evaluaciju

Koristimo jedinstvenu funkciju `evaluate_model` koja vraća RMSE, MAE i R² na log-transformiranim vrijednostima.

In [ ]:
def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    return {'model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

results = []

## 3. Model 1 — Linearna regresija (baseline)

Treniramo linearnu regresiju i Ridge regresiju (regularizacija L2, α=10) sa standardiziranim značajkama. Ovo je naš bazni model s kojim uspoređujemo složenije pristupe.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

# Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_val_scaled)
results.append(evaluate_model('Linear Regression', y_val, lr_pred))

# Ridge
ridge = Ridge(alpha=10)
ridge.fit(X_train_scaled, y_train)
ridge_pred = ridge.predict(X_val_scaled)
results.append(evaluate_model('Ridge (α=10)', y_val, ridge_pred))

print('Linear Regression:', results[-2])
print('Ridge:            ', results[-1])

## 4. Model 2 — XGBoost

Gradient boosting model koji kombinira više slabih stabala odlučivanja. Treniramo s 500 stabala i learning rate 0.05 kako bismo izbjegli preučenje.

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbosity=0
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

xgb_pred = xgb_model.predict(X_val)
results.append(evaluate_model('XGBoost', y_val, xgb_pred))
print('XGBoost:', results[-1])

In [ ]:
# Feature importance — top 15
feat_imp = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
feat_imp = feat_imp.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='barh')
plt.gca().invert_yaxis()
plt.title('XGBoost — top 15 značajki po važnosti')
plt.xlabel('Važnost značajke')
plt.tight_layout()
plt.savefig('../output/figures/xgb_feature_importance.png', bbox_inches='tight')
plt.show()

## 5. Model 3 — MLP Neuronska mreža (Keras/TensorFlow)

Višeslojna perceptronska mreža s dropout regularizacijom i EarlyStopping kako bi se spriječilo preučenje. Arhitektura: 256 → 128 → 64 → 1.

In [ ]:
def build_mlp(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

mlp = build_mlp(X_train_scaled.shape[1])
mlp.summary()

In [ ]:
early_stop = EarlyStopping(patience=15, restore_best_weights=True, monitor='val_loss')

history = mlp.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train loss')
axes[0].plot(history.history['val_loss'], label='Val loss')
axes[0].set_title('MLP — krivulje gubitka (MSE)')
axes[0].set_xlabel('Epoha')
axes[0].set_ylabel('MSE')
axes[0].legend()

axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_title('MLP — krivulje MAE')
axes[1].set_xlabel('Epoha')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.savefig('../output/figures/mlp_learning_curve.png', bbox_inches='tight')
plt.show()

In [ ]:
mlp_pred = mlp.predict(X_val_scaled).flatten()
results.append(evaluate_model('MLP', y_val, mlp_pred))
print('MLP:', results[-1])

## 6. SHAP analiza (objašnjivost XGBoost modela)

SHAP (SHapley Additive exPlanations) vrijednosti kvantificiraju doprinos svake značajke pojedinoj predikciji. Koristimo TreeExplainer koji je optimiziran za stabla odlučivanja.

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_val)

In [ ]:
# SHAP summary bar
plt.figure()
shap.summary_plot(shap_values, X_val, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig('../output/figures/shap_summary_bar.png', bbox_inches='tight')
plt.show()

In [ ]:
# SHAP beeswarm
plt.figure()
shap.summary_plot(shap_values, X_val, show=False)
plt.tight_layout()
plt.savefig('../output/figures/shap_beeswarm.png', bbox_inches='tight')
plt.show()

## 7. Usporedba svih modela

Prikazujemo skupljene metrike svih modela u tablici i grupanom stupčastom grafu.

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.set_index('model')
print(results_df.to_string())

In [ ]:
# Grouped bar chart
metrics = ['RMSE', 'MAE', 'R2']
x = np.arange(len(results_df))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, metric in enumerate(metrics):
    ax.bar(x + i * width, results_df[metric], width, label=metric)

ax.set_title('Usporedba modela — RMSE / MAE / R²')
ax.set_xticks(x + width)
ax.set_xticklabels(results_df.index, rotation=15, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig('../output/figures/model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
results_df.to_csv('../output/results/model_results.csv')
print('Rezultati spremljeni u output/results/model_results.csv')